# AMR Double-Sphere Analytic Comparison

This notebook reads a RAMSES `grav` output with `miniramses`, finds the AMR refinement structure directly from the leaf cells, and evaluates the analytic double-sphere solution at the same AMR cell centers.

The workflow is:

1. Read one `grav` output with `ram.rd_cell`.
2. Build a slice through the AMR leaf cells.
3. Inspect the refinement regions present in that slice.
4. Evaluate the analytic double-sphere potential and forces at the same AMR leaf-cell centers.
5. Visualize simulation, analytic, and error maps on the AMR mesh.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append("../utils/py")
import miniramses as ram

plt.rcParams["font.family"] = "serif"
plt.rcParams["figure.dpi"] = 180
plt.rcParams.update({
    "font.size": 14,
    "legend.fontsize": 12,
})

G = 1.0


In [ ]:
# -----------------------------
# User configuration
# -----------------------------
path = ".."          # directory that contains output_0000*
prefix = "grav"
nout = 2             # output number to inspect

# Slice definition: keep cells whose volume intersects this plane.
slice_axis = "z"
slice_center = 1.0   # code units

# Optional plotting box [xmin, xmax, ymin, ymax] in the slice plane.
plot_box = None

# Double-sphere parameters in code units.
rho = 1.0
sphere1_center = np.array([0.7, 1.0, 1.0])
sphere2_center = np.array([1.2, 1.0, 1.0])
sphere1_radius = 0.1
sphere2_radius = 0.2


In [ ]:
# -----------------------------
# Analytic double-sphere solution
# -----------------------------
def _uniform_sphere_mass(radius, density=rho):
    return (4.0 / 3.0) * np.pi * radius**3 * density


def _sphere_offsets(x, y, z, center):
    dx = x - center[0]
    dy = y - center[1]
    dz = z - center[2]
    r = np.sqrt(dx**2 + dy**2 + dz**2)
    r = np.maximum(r, 1.0e-14)
    return dx, dy, dz, r


def phi_uniform_sphere(r, radius, mass):
    phi_out = -G * mass / r
    phi_in = -1.5 * G * mass / radius + 0.5 * G * mass * r**2 / radius**3
    return np.where(r >= radius, phi_out, phi_in)


def force_component_uniform_sphere(delta, r, radius, mass):
    f_out = -G * mass * delta / r**3
    f_in = -G * mass * delta / radius**3
    return np.where(r >= radius, f_out, f_in)


def analytic_double_sphere(x, y, z):
    """Return analytic phi, fx, fy, fz, |f| at arbitrary AMR cell centers."""
    m1 = _uniform_sphere_mass(sphere1_radius)
    m2 = _uniform_sphere_mass(sphere2_radius)

    dx1, dy1, dz1, r1 = _sphere_offsets(x, y, z, sphere1_center)
    dx2, dy2, dz2, r2 = _sphere_offsets(x, y, z, sphere2_center)

    phi = phi_uniform_sphere(r1, sphere1_radius, m1) + phi_uniform_sphere(r2, sphere2_radius, m2)
    fx = force_component_uniform_sphere(dx1, r1, sphere1_radius, m1) + force_component_uniform_sphere(dx2, r2, sphere2_radius, m2)
    fy = force_component_uniform_sphere(dy1, r1, sphere1_radius, m1) + force_component_uniform_sphere(dy2, r2, sphere2_radius, m2)
    fz = force_component_uniform_sphere(dz1, r1, sphere1_radius, m1) + force_component_uniform_sphere(dz2, r2, sphere2_radius, m2)
    fmag = np.sqrt(fx**2 + fy**2 + fz**2)

    return {
        "phi": phi,
        "fx": fx,
        "fy": fy,
        "fz": fz,
        "f": fmag,
    }


In [ ]:
# -----------------------------
# AMR loading and slicing helpers
# -----------------------------
AXIS_TO_INDEX = {"x": 0, "y": 1, "z": 2}


def load_grav_cells(nout=nout, path=path, prefix=prefix):
    c = ram.rd_cell(str(nout), path=path, prefix=prefix)
    info = ram.rd_info(str(nout), path=path)
    return c, info


def slice_cells(c, axis=slice_axis, center=slice_center, levels=None):
    axis = axis.lower()
    if axis not in AXIS_TO_INDEX:
        raise ValueError(f"axis must be one of {tuple(AXIS_TO_INDEX)}, got {axis!r}")

    axis_idx = AXIS_TO_INDEX[axis]
    mask = np.abs(c.x[axis_idx] - center) <= c.dx / 2.0

    if levels is not None:
        mask &= np.isin(c.level, np.asarray(levels))

    data = {
        "mask": mask,
        "x3d": c.x[0][mask],
        "y3d": c.x[1][mask],
        "z3d": c.x[2][mask],
        "dx": c.dx[mask],
        "level": c.level[mask],
        "level_plot": c.level[mask] + 1,
        "phi_sim": c.g[0][mask],
        "fx_sim": c.g[1][mask],
        "fy_sim": c.g[2][mask],
        "fz_sim": c.g[3][mask],
    }
    data["f_sim"] = np.sqrt(data["fx_sim"]**2 + data["fy_sim"]**2 + data["fz_sim"]**2)

    analytic = analytic_double_sphere(data["x3d"], data["y3d"], data["z3d"])
    for key, value in analytic.items():
        data[f"{key}_ana"] = value
        sim_key = f"{key}_sim"
        if sim_key in data:
            with np.errstate(divide="ignore", invalid="ignore"):
                data[f"{key}_relerr"] = np.where(np.abs(value) > 0.0, (data[sim_key] - value) / value, np.nan)

    if axis == "x":
        data["u"] = data["y3d"]
        data["v"] = data["z3d"]
        data["plane_axes"] = ("y", "z")
    elif axis == "y":
        data["u"] = data["x3d"]
        data["v"] = data["z3d"]
        data["plane_axes"] = ("x", "z")
    else:
        data["u"] = data["x3d"]
        data["v"] = data["y3d"]
        data["plane_axes"] = ("x", "y")

    return data


def slice_dataframe(s):
    return pd.DataFrame({
        "x": s["x3d"],
        "y": s["y3d"],
        "z": s["z3d"],
        "u": s["u"],
        "v": s["v"],
        "dx": s["dx"],
        "level": s["level"],
        "level_plot": s["level_plot"],
        "phi_sim": s["phi_sim"],
        "phi_ana": s["phi_ana"],
        "phi_relerr": s["phi_relerr"],
        "fx_sim": s["fx_sim"],
        "fy_sim": s["fy_sim"],
        "fz_sim": s["fz_sim"],
        "f_sim": s["f_sim"],
        "f_ana": s["f_ana"],
        "f_relerr": s["f_relerr"],
    })


In [ ]:
# -----------------------------
# Refinement-region summaries
# -----------------------------
def refinement_summary(s):
    records = []
    for lev in np.sort(np.unique(s["level"])):
        keep = s["level"] == lev
        if not np.any(keep):
            continue
        u = s["u"][keep]
        v = s["v"][keep]
        dx = s["dx"][keep]
        records.append({
            "level": int(lev + 1),
            "ncell": int(np.count_nonzero(keep)),
            "dx": float(dx[0]),
            "u_min": float(np.min(u - dx / 2.0)),
            "u_max": float(np.max(u + dx / 2.0)),
            "v_min": float(np.min(v - dx / 2.0)),
            "v_max": float(np.max(v + dx / 2.0)),
        })
    return pd.DataFrame(records)


def show_refinement_summary(s):
    df = refinement_summary(s)
    if df.empty:
        print("No cells intersect this slice.")
        return df
    display(df)
    return df


In [ ]:
# -----------------------------
# Visualization helpers
# -----------------------------
def show_amr_field(s, field, *, levels=None, box=plot_box, npix=900, grid=False,
                   cmap="magma", log=False, vmin=None, vmax=None):
    keep = np.ones_like(s["level"], dtype=bool)
    if levels is not None:
        keep &= np.isin(s["level"], np.asarray(levels))

    if not np.any(keep):
        raise ValueError("No slice cells survive the requested level filter.")

    return ram.visu(
        s["u"][keep],
        s["v"][keep],
        s["dx"][keep],
        s[field][keep],
        s["level_plot"][keep],
        npix=npix,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        box=box,
        grid=grid,
        log=log,
    )


def plot_refinement_map(s, box=plot_box, npix=900):
    return show_amr_field(
        s,
        "level_plot",
        box=box,
        npix=npix,
        grid=True,
        cmap="viridis",
    )


def compare_scalar_field(s, quantity="phi", box=plot_box, npix=900, log=False):
    sim_key = f"{quantity}_sim"
    ana_key = f"{quantity}_ana"
    err_key = f"{quantity}_relerr"

    if sim_key not in s or ana_key not in s or err_key not in s:
        raise KeyError(f"Unknown quantity {quantity!r}")

    sim_vals = s[sim_key]
    ana_vals = s[ana_key]
    err_vals = s[err_key]

    finite_vals = np.concatenate([sim_vals[np.isfinite(sim_vals)], ana_vals[np.isfinite(ana_vals)]])
    vmin = float(np.min(finite_vals))
    vmax = float(np.max(finite_vals))

    print(f"Simulation {quantity}")
    show_amr_field(s, sim_key, box=box, npix=npix, cmap="magma", log=log, vmin=vmin, vmax=vmax)

    print(f"Analytic {quantity}")
    show_amr_field(s, ana_key, box=box, npix=npix, cmap="magma", log=log, vmin=vmin, vmax=vmax)

    vmax_err = float(np.nanmax(np.abs(err_vals)))
    if not np.isfinite(vmax_err) or vmax_err == 0.0:
        vmax_err = 1.0e-16

    print(f"Relative error in {quantity}")
    show_amr_field(s, err_key, box=box, npix=npix, cmap="RdBu_r", log=False, vmin=-vmax_err, vmax=vmax_err)


def plot_level_error_stats(s, quantity="phi"):
    err_key = f"{quantity}_relerr"
    levels = np.sort(np.unique(s["level"]))
    levels_plot = levels + 1
    means = []
    maxes = []

    for lev in levels:
        keep = s["level"] == lev
        err = np.abs(s[err_key][keep])
        err = err[np.isfinite(err)]
        means.append(np.nan if err.size == 0 else np.mean(err))
        maxes.append(np.nan if err.size == 0 else np.max(err))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(levels_plot, means, "o-", label="mean |relative error|")
    ax.plot(levels_plot, maxes, "s-", label="max |relative error|")
    ax.set_xlabel("AMR level")
    ax.set_ylabel(f"|relative error| in {quantity}")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


def plot_midline_profile(s, quantity="phi", axis_in_plane="u", coord=None, atol=None):
    if axis_in_plane not in {"u", "v"}:
        raise ValueError("axis_in_plane must be 'u' or 'v'")

    transverse = "v" if axis_in_plane == "u" else "u"
    label_main, label_transverse = s["plane_axes"] if axis_in_plane == "u" else s["plane_axes"][::-1]

    if coord is None:
        coord = 0.5 * (np.min(s[transverse]) + np.max(s[transverse]))
    if atol is None:
        atol = np.min(s["dx"]) / 2.0

    keep = np.abs(s[transverse] - coord) <= atol
    if not np.any(keep):
        transverse_values = s[transverse]
        nearest = transverse_values[np.argmin(np.abs(transverse_values - coord))]
        coord = float(nearest)
        keep = np.abs(s[transverse] - coord) <= np.min(s["dx"]) / 2.0
    if not np.any(keep):
        raise ValueError("No slice cells intersect the requested profile band.")

    order = np.argsort(s[axis_in_plane][keep])
    x = s[axis_in_plane][keep][order]
    sim = s[f"{quantity}_sim"][keep][order]
    ana = s[f"{quantity}_ana"][keep][order]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x, ana, "k-", lw=1.5, label="analytic")
    ax.plot(x, sim, "o", ms=3, alpha=0.8, label="AMR cells")
    ax.set_xlabel(label_main)
    ax.set_ylabel(quantity)
    ax.set_title(f"{quantity} profile near {label_transverse} = {coord:.4f}")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


In [ ]:
# Load one grav output and inspect the available AMR levels.
c, info = load_grav_cells()
print(f"Loaded output_{int(nout):05d} from {Path(path).resolve()}")
print(f"ndim = {c.ndim}, ncell = {c.ncell}")
print(f"levelmin = {info.levelmin}, nlevelmax = {info.nlevelmax}")
print("levels present:", np.unique(c.level + 1))


In [ ]:
# Build the AMR slice that intersects the requested plane.
s = slice_cells(c, axis=slice_axis, center=slice_center)
print(f"slice axis   : {slice_axis}")
print(f"slice center : {slice_center}")
print(f"slice cells  : {len(s['level'])}")
print("slice levels:", np.unique(s["level_plot"]))


In [ ]:
# Summarize the refinement regions present in the slice.
summary_df = show_refinement_summary(s)
summary_df


In [ ]:
# AMR refinement map for the slice.
# The color encodes the leaf-cell AMR level, so this shows the refinement regions directly.
plot_refinement_map(s, box=plot_box, npix=900)


In [ ]:
# Compare simulation and analytic potential on the same AMR leaf cells.
compare_scalar_field(s, quantity="phi", box=plot_box, npix=900, log=False)


In [ ]:
# Compare force magnitude as well, if you want to inspect errors away from the sphere centers.
compare_scalar_field(s, quantity="f", box=plot_box, npix=900, log=True)


In [ ]:
# Error statistics by AMR level.
plot_level_error_stats(s, quantity="phi")
plot_level_error_stats(s, quantity="f")


In [ ]:
# Midline profile across the slice.
# For the default z-slice this means a profile in x near the middle of y.
plot_midline_profile(s, quantity="phi", axis_in_plane="u")
